In [1]:
from debugpy.launcher import output
# Load env variables and create API client
from dotenv import load_dotenv

load_dotenv()

from anthropic import Anthropic
client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    for block in message.content:
        if block.type == "text":
            return block.text

In [31]:
# Opcion recomendada ya que el curso usa stop_sequences que esta deprecado
from anthropic.types import ToolParam, tool_use_block

dataset_tool_schema = ToolParam({
    "name": "generate_dataset",
    "description": "Genera una lista de casos de prueba para evaluar el prompt.",
    "input_schema": {
        "type": "object",
        "properties": {
            "tasks": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "task": {"type": "string"},
                        "format": {"type": "string", "enum": ["json", "python", "regex"]}
                    },
                    "required": ["task", "format"]
                }
            }
        },
        "required": ["tasks"]
    }
})

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "json" or "python" or "regex"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
    """
    messages = []
    add_user_message(messages, prompt)

    response = client.messages.create(
        model=model,
        max_tokens=1024,
        messages=messages,
        tools=[dataset_tool_schema],
        tool_choice={"type": "tool", "name": "generate_dataset"}
    )
    tool_use_block = next(b for b in response.content if b.type == "tool_use")
    return tool_use_block.input["tasks"]

In [32]:
dataset = generate_dataset()

print(dataset)


[{'task': "Parse an AWS S3 bucket name and region from an S3 URI (e.g., 's3://my-bucket/path/to/object') and return it as a structured format", 'format': 'python'}, {'task': 'Create a JSON policy document for an AWS IAM role that allows read-only access to a specific S3 bucket', 'format': 'json'}, {'task': 'Create a regex pattern that matches valid AWS Lambda function names (alphanumeric characters, hyphens, and underscores, 1-64 characters)', 'format': 'regex'}]


In [33]:
import json

dataset = generate_dataset()
# generate_dataset() return ["task", "format"] but we need a list of dicts with "task" and "format" keys
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)




In [44]:
CODE_SCHEMA = {
    "type": "object",
    "properties": {
        "code": {"type": "string"},
    },
    "required": ["code"],
    "additionalProperties": False,
}

def run_prompt(test_case):
    prompt = f"""
Please solve the following task:
{test_case['task']}

* The solution must be valid {test_case['format']} and nothing else
* Put only the solution in the "code" field
* Do not add any comments, commentary or explanation
* Do not wrap the solution in markdown code fences
"""
    messages = []
    add_user_message(messages, prompt)
    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        output_config={
            "format": {
                "type": "json_schema",
                "schema": CODE_SCHEMA,
            }
        },
    )
    text = next(block.text for block in response.content if block.type == "text")
    return json.loads(text)["code"]

In [35]:
import json
from typing import Any

EVAL_SCHEMA = {
    "type": "object",
    "properties": {
        "strengths": {"type": "array", "items": {"type": "string"}},
        "weaknesses": {"type": "array", "items": {"type": "string"}},
        "reasoning": {"type": "string"},
        "score": {"type": "integer"},
    },
    "required": ["strengths", "weaknesses", "reasoning", "score"],
    "additionalProperties": False,
}

EVAL_PROMPT = """You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {task}
Solution: {solution}

Provide your evaluation with:
- strengths: 1-3 key strengths
- weaknesses: 1-3 key areas for improvement
- reasoning: a concise explanation of your assessment
- score: a number between 1 and 10
"""


def grade_by_model(test_case, output):
    eval_prompt = EVAL_PROMPT.format(task=test_case["task"], solution=output)

    messages = []
    add_user_message(messages, eval_prompt)

    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        output_config={
            "format": {
                "type": "json_schema",
                "schema": EVAL_SCHEMA,
            }
        },
    )

    # Mismo patrón que tu referencia: buscar el bloque de texto
    text = next(block.text for block in response.content if block.type == "text")

    result: Any = json.loads(text)
    result["score"] = max(1, min(10, result["score"]))  # el schema no valida rangos
    return result

In [52]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    elif format == "regex":
        return validate_regex(response)

In [53]:
def run_test_case(test_case):
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2
    return{
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

In [54]:
from statistics import mean

def run_eval(dataset):
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [55]:
import json
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.0


In [56]:
print(json.dumps(results, indent=2))

[
  {
    "output": "arn:aws:s3:::([a-z0-9._-]+)(?:/.*)?$",
    "test_case": {
      "task": "Parse an AWS S3 bucket ARN and extract the bucket name using regex",
      "format": "regex"
    },
    "score": 8.0,
    "reasoning": "The regex demonstrates solid understanding of S3 ARN structure (arn:aws:s3:::bucket-name) and uses a capture group appropriately. However, it has a critical limitation in only matching lowercase characters [a-z0-9._-], which excludes uppercase letters that AWS S3 buckets can contain. While the optional path suffix handling is correct, the case-sensitivity issue significantly reduces practical usability. The solution works for lowercase bucket names but fails on common real-world cases with uppercase characters."
  },
  {
    "output": "import json\nfrom typing import Union, Dict, List, Any\n\ndef validate_iam_policy(policy: Union[str, Dict]) -> Dict[str, Any]:\n    \"\"\"\n    Validates an AWS IAM policy JSON structure for required fields.\n    \n    Args:\n  

# Comentarios adicionales

In [ ]:
# Opcion alternativa usando re y json
# Explicacion de claude: https://claude.ai/share/2b9656fd-240c-46ff-80fc-787dda2e3db9
import json
import re

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
    """
    messages = []
    # El curso usa stop_sequences que esta deprecado
    add_user_message(messages, prompt)
    text = chat(messages)

    # Extrae el contenido entre ```json y ``` si existe
    match = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    if match:
        text = match.group(1)

    return json.loads(text.strip())